# WordNet with NLTK

This notebook demonstrates how to use NLTK's interface to WordNet, a lexical database for English. It covers lookups, synsets, lemmas, and similarity measures.

## 1. Setup

First, we import the necessary modules and download the WordNet data.

In [1]:
import nltk
from nltk.corpus import wordnet as wn

# Ensure necessary data is downloaded
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 2. Words

Look up a word using `synsets()`. This function returns a list of Synset objects. You can optionally constrain by part of speech (POS).

In [2]:
# Look up a word
wn.synsets('dog')

[Synset('dog.n.01'),
 Synset('frump.n.01'),
 Synset('dog.n.03'),
 Synset('cad.n.01'),
 Synset('frank.n.02'),
 Synset('pawl.n.01'),
 Synset('andiron.n.01'),
 Synset('chase.v.01')]

In [3]:
# Constrain by POS
wn.synsets('dog', pos=wn.VERB)

[Synset('chase.v.01')]

## 3. Synsets

A synset is identified with a 3-part name of the form: `word.pos.nn`.
We can access the definition and examples for a synset.

In [4]:
synset = wn.synset('dog.n.01')
print("Name:", synset.name())
print("Definition:", synset.definition())
print("Examples:", synset.examples())

Name: dog.n.01
Definition: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
Examples: ['the dog barked all night']


In [5]:
# Get lemmas for a synset
synset.lemmas()

[Lemma('dog.n.01.dog'),
 Lemma('dog.n.01.domestic_dog'),
 Lemma('dog.n.01.Canis_familiaris')]

In [6]:
# Get lemma names from lemmas
[str(lemma.name()) for lemma in synset.lemmas()]

['dog', 'domestic_dog', 'Canis_familiaris']

### Synset Relations

We can explore relations between synsets, such as hypernyms (more general) and hyponyms (more specific).

In [7]:
dog = wn.synset('dog.n.01')
print("Hypernyms:", dog.hypernyms())
print("Hyponyms (first 5):", sorted(dog.hyponyms())[:5])
print("Member Holonyms:", dog.member_holonyms())
print("Root Hypernyms:", dog.root_hypernyms())

Hypernyms: [Synset('domestic_animal.n.01'), Synset('canine.n.02')]
Hyponyms (first 5): [Synset('basenji.n.01'), Synset('corgi.n.01'), Synset('cur.n.01'), Synset('dalmatian.n.02'), Synset('great_pyrenees.n.01')]
Member Holonyms: [Synset('pack.n.06'), Synset('canis.n.01')]
Root Hypernyms: [Synset('entity.n.01')]


In [8]:
# Lowest common hypernym
cat = wn.synset('cat.n.01')
dog.lowest_common_hypernyms(cat)

[Synset('carnivore.n.01')]

## 4. Lemmas

Lemmas represent a specific sense of a specific word. They can have their own relations, like antonyms.

In [9]:
eat = wn.lemma('eat.v.03.eat')
print(eat)
print("Key:", eat.key())
print("Count:", eat.count())

Lemma('feed.v.06.eat')
Key: eat%2:34:02::
Count: 4


In [10]:
# Antonyms are defined on lemmas, not synsets
good = wn.synset('good.a.01')
print("Synset:", good)
# good.antonyms() # This would raise an AttributeError
print("Antonyms:", good.lemmas()[0].antonyms())

Synset: Synset('good.a.01')
Antonyms: [Lemma('bad.a.01.bad')]


## 5. Similarity

WordNet can calculate similarity scores between synsets.

In [11]:
dog = wn.synset('dog.n.01')
cat = wn.synset('cat.n.01')

hit = wn.synset('hit.v.01')
slap = wn.synset('slap.v.01')

### Path Similarity
Return a score denoting how similar two word senses are, based on the shortest path that connects the senses in the is-a (hypernym/hyponym) taxonomy.

In [12]:
print("Dog-Cat Path Similarity:", dog.path_similarity(cat))
print("Hit-Slap Path Similarity:", hit.path_similarity(slap))

Dog-Cat Path Similarity: 0.2
Hit-Slap Path Similarity: 0.14285714285714285


### Leacock-Chodorow Similarity
Return a score denoting how similar two word senses are, based on the shortest path that connects the senses (as above) and the maximum depth of the taxonomy in which the senses occur.

In [13]:
print("Dog-Cat LCH Similarity:", dog.lch_similarity(cat))
print("Hit-Slap LCH Similarity:", hit.lch_similarity(slap))

Dog-Cat LCH Similarity: 2.0281482472922856
Hit-Slap LCH Similarity: 1.3121863889661687


### Wu-Palmer Similarity
Return a score denoting how similar two word senses are, based on the depth of the two senses in the taxonomy and that of their Least Common Subsumer (most specific ancestor node).

In [14]:
print("Dog-Cat Wu-Palmer Similarity:", dog.wup_similarity(cat))
print("Hit-Slap Wu-Palmer Similarity:", hit.wup_similarity(slap))

Dog-Cat Wu-Palmer Similarity: 0.8571428571428571
Hit-Slap Wu-Palmer Similarity: 0.25


### Information Content based Similarity
These measures use Information Content (IC) from a corpus.

In [15]:
from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')
semcor_ic = wordnet_ic.ic('ic-semcor.dat')

# Resnik Similarity
print("Resnik (Brown):", dog.res_similarity(cat, brown_ic))

# Jiang-Conrath Similarity
print("Jiang-Conrath (Brown):", dog.jcn_similarity(cat, brown_ic))

# Lin Similarity
print("Lin (Semcor):", dog.lin_similarity(cat, semcor_ic))

[nltk_data] Downloading package wordnet_ic to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Unzipping corpora/wordnet_ic.zip.


Resnik (Brown): 7.911666509036577
Jiang-Conrath (Brown): 0.4497755285516739
Lin (Semcor): 0.8863288628086228


## 6. Access to all Synsets

We can iterate over all synsets, or synsets of a specific POS.

In [16]:
for synset in list(wn.all_synsets('n'))[:5]:
    print(synset)

Synset('entity.n.01')
Synset('physical_entity.n.01')
Synset('abstraction.n.06')
Synset('thing.n.12')
Synset('object.n.01')


## 7. Morphy

Morphy attempts to find the base form of a word.

In [17]:
print(wn.morphy('denied', wn.NOUN))
print(wn.morphy('denied', wn.VERB))
print(wn.morphy('dogs'))
print(wn.morphy('churches'))

None
deny
dog
church
